# Validating models correctly

_Doing ML for Real — the skills that matter_

**Your score is a guess about the future. Most ways of computing it lie — here is how to make it honest.**

A model's score on its own training data is meaningless — it can memorize. We want the generalization error: how it does on fresh data drawn from the same world.

       We never have the whole future, so we estimate generalization error by holding data out. Cross-validation reuses the data efficiently by rotating which slice is held out. But the estimate is only honest if the held-out slice is truly independent of what trained the model.

---

This notebook is a practice scaffold for the **Validating models correctly** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Load the data inside a leak-proof Pipeline

The first defence against a dishonest score is to keep **all** preprocessing inside a `Pipeline`. When cross-validation hands a training fold to the pipeline, the `StandardScaler` is fit on that fold only — never on the held-out fold. That way the scaler never "peeks" at the data it will be scored on, so there is no leakage by construction.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True)

# ALL preprocessing lives inside the Pipeline, so the scaler is fit on the
# TRAINING fold only, never on the held-out fold. No leakage by construction.
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("clf",   LogisticRegression(max_iter=5000)),
])

### Step 2 — Pick the right splitter, then score

How you slice the data has to match its structure. `StratifiedKFold` keeps the class ratio in every fold (important when the classes are imbalanced). `GroupKFold` makes sure one entity — a patient or user — never lands on both sides of a split. `TimeSeriesSplit` trains on the past and tests on the future, never shuffling. Here the data is i.i.d. and slightly imbalanced, so we score with `StratifiedKFold`.

In [ ]:
from sklearn.model_selection import (
    StratifiedKFold, GroupKFold, TimeSeriesSplit, cross_val_score,
)

# StratifiedKFold: keeps the class ratio in every fold (imbalance).
strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
# GroupKFold: an entity (patient/user) never straddles the split.
group = GroupKFold(n_splits=5)
# TimeSeriesSplit: train on the past, test on the future (no shuffling).
ts = TimeSeriesSplit(n_splits=5)

scores = cross_val_score(pipe, X, y, cv=strat, scoring="roc_auc")
print("per-fold AUC:", np.round(scores, 3))

### Step 3 — Tune honestly with nested cross-validation

If you tune a hyperparameter and report that same tuned score, the number is optimistic — you picked the best of many trials on the same folds. **Nested CV** fixes this: an *inner* `GridSearchCV` chooses `C`, and an *outer* CV loop scores the whole tune-then-fit procedure on folds the inner search never saw. The outer score is what honestly generalizes.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Inner search picks C; outer loop scores the chosen procedure on unseen folds.
grid = {"clf__C": [0.01, 0.1, 1, 10]}
inner = StratifiedKFold(n_splits=4, shuffle=True, random_state=1)
outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

search = GridSearchCV(pipe, grid, cv=inner, scoring="roc_auc")
nested = cross_val_score(search, X, y, cv=outer, scoring="roc_auc")  # outer loop
print("nested-CV AUC: %.3f" % nested.mean())

### Step 4 — Report an interval and compare models fairly

A single number hides its own uncertainty. A **bootstrap** resamples the fold scores to build a 95% confidence interval around the estimate. And to decide whether one model truly beats another, we use a **paired** test: train both on the *same* split and run McNemar's test on their disagreements — the cases where one is right and the other wrong. The p-value tells you whether the gap is real or just noise.

In [ ]:
# Bootstrap 95% confidence interval on the estimate (not a single number).
def bootstrap_ci(fold_scores, B=10000, seed=0):
    rng = np.random.default_rng(seed)
    means = [rng.choice(fold_scores, size=len(fold_scores), replace=True).mean()
             for _ in range(B)]
    return np.percentile(means, [2.5, 97.5])

lo, hi = bootstrap_ci(nested)
print("nested-CV AUC %.3f  95%% CI [%.3f, %.3f]" % (nested.mean(), lo, hi))

# Compare two models with McNemar's paired test on one held-out test set.
from sklearn.model_selection import train_test_split
from statsmodels.stats.contingency_tables import mcnemar  # paired test

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

# Model A: the default pipeline.  Model B: same pipeline but a heavier C=0.01.
pA = pipe.fit(Xtr, ytr).predict(Xte)
pipe_b = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(C=0.01, max_iter=5000)),
])
pB = pipe_b.fit(Xtr, ytr).predict(Xte)

b = int(((pA == yte) & (pB != yte)).sum())   # A right, B wrong
c = int(((pB == yte) & (pA != yte)).sum())   # B right, A wrong
print("McNemar:", mcnemar([[0, b], [c, 0]], exact=False, correction=True))

## Visualize the data & results

_On real data with a grouped (entity) structure, how much does naive random K-fold INFLATE the score versus a group-aware split? The gap is leakage._

### Step 1 — Manufacture grouped (non-independent) data

To *show* leakage we need rows that are not independent. We take 80 real tumours and treat each as a "patient", then generate several near-identical "scans" per patient by adding tiny noise. Now many rows share a hidden identity — exactly the structure that random K-fold mishandles.

In [ ]:
from sklearn.model_selection import KFold, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier

d = load_breast_cancer()
Xfull, yfull = d.data, d.target
rng = np.random.default_rng(0)

# Build a GROUP (entity) index: 80 real tumours act as "patients",
# each contributing 5-8 near-identical "scans" (original row + tiny noise).
# This makes rows NON-independent — the situation that breaks random K-fold.
n_groups = 80
base = rng.choice(len(yfull), size=n_groups, replace=False)

Xg, yg, groups = [], [], []
for g, bi in enumerate(base):
    n_scans = rng.integers(5, 9)
    for _ in range(n_scans):
        noisy_twin = Xfull[bi] + rng.normal(0, 0.02, Xfull.shape[1]) * Xfull[bi]
        Xg.append(noisy_twin)
        yg.append(yfull[bi])
        groups.append(g)

Xg = np.array(Xg)
yg = np.array(yg)
groups = np.array(groups)
print("rows:", len(yg), " patients:", n_groups)

### Step 2 — Score it two ways and measure the leakage gap

We run the *same* 1-nearest-neighbour pipeline under two splitters. Random `KFold` ignores the groups, so a near-identical twin of each test row sits in the training fold — an optimistic, dishonest score. `GroupKFold` keeps every patient entirely on one side, giving the honest score. The difference between the two is the leakage.

In [ ]:
pipe_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))

# OPTIMISTIC: random K-fold ignores groups -> a twin of each test row is in train.
opt = cross_val_score(pipe_knn, Xg, yg, cv=KFold(5, shuffle=True, random_state=0))

# HONEST: GroupKFold keeps each patient entirely on one side.
hon = cross_val_score(pipe_knn, Xg, yg, groups=groups, cv=GroupKFold(5))

print("optimistic (random KFold):", round(opt.mean(), 3))  # ~1.000
print("honest (GroupKFold)      :", round(hon.mean(), 3))  # ~0.929
print("leakage gap              :", round(opt.mean() - hon.mean(), 3))  # ~0.071

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your dataset has 50 patients, each with about 6 MRI scans (rows). You run KFold(shuffle=True) and report 0.97 accuracy. Your colleague says the number is not trustworthy. Why, and how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice the rows are not independent: 6 scans from the same patient are near-duplicates. — _Random shuffling scatters one patient's scans across both train and test folds._
- Name the leak: a scan of patient P is in test while another scan of the SAME patient P is in train. — _The model recognizes the patient, not the disease, so the test score measures memorization. That is why 0.97 is inflated._
- Switch to GroupKFold with groups = patient id, so every scan of a patient lands entirely in train OR test. — _Now the test patients are genuinely unseen, and the score honestly estimates how the model does on a NEW patient._

**Answer:** The rows are grouped by patient, so random K-fold lets a patient's scans straddle train and test (leakage) — the model recognizes patients, inflating the score. Fix: GroupKFold with groups = patient id, so no patient appears on both sides. Expect a lower but honest number.

</details>

**Problem 2.** You used GridSearchCV to pick the best regularizer, then reported that GridSearchCV's best cross-validation score as your final accuracy. What is wrong, and what is the correct protocol?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize that the reported number is the MAX over many hyperparameter trials on the same folds. — _Taking the best of many noisy scores is biased upward even if every setting were equally good — optimistic selection bias._
- Wrap the GridSearchCV inside an OUTER cross-validation loop (nested CV). — _The outer folds score the whole tune-then-fit procedure on data the inner grid search never saw._
- Report the outer-loop score (with an interval), not the inner best score. — _The outer score honestly estimates what your full model-building recipe achieves on fresh data._

**Answer:** Reporting GridSearchCV's own best score tunes and evaluates on the same folds, so it is optimistically biased (the max of many trials). Correct protocol: nested CV — an inner GridSearchCV to choose hyperparameters, an outer CV loop to score the chosen procedure on unseen folds. Report the outer score plus a confidence interval.

</details>

**Problem 3.** Two models score 0.903 and 0.911 on your data. A teammate declares the second one "better". How do you decide whether that 0.008 gap is real?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Refuse to compare two bare means from possibly different runs. — _A 0.008 gap can easily be split luck; and if the scores came from different folds, model and split are confounded._
- Score both models on the SAME folds (or the same held-out test set), example by example. — _A paired comparison removes the split as a variable: each fold contributes a difference, not two independent numbers._
- Run McNemar's test on the test-set disagreements (b = A right/B wrong, c = B right/A wrong), or a paired CV test, and check the p-value. — _It tells you whether the difference is statistically real or within noise. Only then declare a winner._

**Answer:** Do not trust the raw 0.008 gap. Evaluate both models on the same folds/test set and run a paired test — McNemar on the disagreement counts (b, c) for a single test set, or a paired CV test. If the p-value is below 0.05 the gap is real; otherwise the models are tied and you should prefer the simpler one.

</details>